In [ ]:
# V5 prerequisite check; no fallback to historical outputs.
from pathlib import Path
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
V5_ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')] if (p.parent/'Data').is_dir()),None)
assert V5_ROOT is not None, 'Permanent Final/Data not found'
required=["05_rgb_dsm_ms_matched_v1/05_harmonized_sensor_registry.csv","05_rgb_dsm_ms_matched_v1/06_all_five_models_matched_unit_metrics.csv","05_rgb_dsm_ms_matched_v1/06_reporting_contract.json","05_rgb_dsm_ms_matched_v1/06_all_106_unit_support.csv","04_unified_rgb_heldout_v5/04_heldout_unit_metrics.csv"]
missing=[str(V5_ROOT/'Outputs'/p) for p in required if not (V5_ROOT/'Outputs'/p).is_file()]
assert not missing, 'Missing native V5 prerequisites: '+str(missing)



# Musanga V5 — NB05bis lightweight acquisition-quality diagnostics

CPU only, no training. Reuse NB04 and NB05 native V5 scores and joint sensor support. Read RGB only, once per footprint, at a fixed 0.5 m physical sampling scale. Test scene-dependent contrast and detail proxies against held-out cover errors; compare MS versus matched RGB control. No new MS pixel reads, registration estimates, spectral texture analysis or quality-based exclusions. Results are exploratory, not measured blur or photogrammetric accuracy.


## 00 — Read-only sources and audit parameters

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely scipy pandas matplotlib scikit-learn statsmodels
from pathlib import Path
from contextlib import ExitStack
from collections import defaultdict
import json,math,os,re,warnings,hashlib,time
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import Affine
from rasterio.features import rasterize
from rasterio.windows import Window
from scipy.ndimage import gaussian_filter, sobel,laplace, distance_transform_edt
from scipy.stats import spearmanr
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),
                        Path('/content/drive/MyDrive/Musanga/Final/V5')) if p.is_dir()),None)
assert ROOT is not None
DATA=ROOT.parent/'Data'; V4=ROOT/'Outputs'
NB05=V4/'05_rgb_dsm_ms_matched_v1'
SPLIT=V4/'01_spatial_cover_folds_v2'
NB02=V4/'02_rgb_circles_plus_plots_v3'
OUT=V4/'05bis_acquisition_quality_v1';FIG=OUT/'figures'
for p in (OUT,FIG):p.mkdir(parents=True,exist_ok=True)
V31=json.loads((ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
UNITS=gpd.read_file(SPLIT/'01_reference_groups_and_folds_v2.gpkg')
UNITS['unit_id']=UNITS.unit_id.astype(str)
OWNERS=pd.read_csv(NB02/'02_reference_mosaic_owners_v2.csv')
OWNERS['unit_id']=OWNERS.unit_id.astype(str)
UNITS=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
                  on=['unit_type','unit_id'],validate='one_to_one')
UNITS=UNITS[UNITS.roi_mosaic_coverage_fraction>=.95].copy()
assert len(UNITS)==106
REG=pd.read_csv(NB05/'05_harmonized_sensor_registry.csv')
assert {'group','band','data_band','path'}.issubset(REG.columns)
MOSAICS={}
for entry in V31['mosaics']:
 if str(entry['file']).endswith('.tif'):
  f=Path(entry['path'])
  if not f.is_file():f=DATA/'Ortho_RGB'/entry['file']
  assert f.is_file(),f
  MOSAICS[entry['file']]=f
assert len(MOSAICS)==6
# Use NB05 exact registry; never mix individual bands from different flights.
GROUP_MOSAIC={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in REG.group.unique():
 if str(group).startswith('Yoko_Carlier2020_MicaSense_'):
  choices=[m for m in MOSAICS if 'yoko' in m.lower()]
  assert len(choices)==1,choices
  GROUP_MOSAIC[group]=choices[0]
assert set(REG.group)==set(GROUP_MOSAIC),set(REG.group)-set(GROUP_MOSAIC)
def bands_for(group):
 q=REG[REG.group.eq(group)]
 assert len(q)==4 and set(q.band)=={'red','green','red_edge','nir'},group
 return {band:(Path(q[q.band.eq(band)].iloc[0].path),
                int(q[q.band.eq(band)].iloc[0].data_band))
         for band in ('red','green','red_edge','nir')}
SITE_FILTER=None # 'MOS' for fast troubleshooting; None for full site comparisons
SAMPLE_M=.50
MAX_DIM=800 # cap per-footprint window on 0.5m grid
BUFFER_M=4
MIN_PIXELS=100
LOCAL_DIAGNOSTIC_M=4
assert SITE_FILTER is None or SITE_FILTER in UNITS.site.unique()
if SITE_FILTER: UNITS=UNITS[UNITS.site.eq(SITE_FILTER)].copy()
print('Reference footprints:',len(UNITS),'| RGB mosaics:',len(MOSAICS),
      '| four-band MS acquisitions:',len(GROUP_MOSAIC),'| output:',OUT)


## 01 — Native source properties and acquisition metadata inventory

Resolutions, imagery channel types and NoData are *measured from raster headers*. Flight speed, exposure, camera parameters, image overlap and acquisition dates remain **not available** unless source metadata actually supply them. A clearly marked manual metadata template is written for optional completion later.

In [ ]:
rows=[]
for name,path in MOSAICS.items():
 with rasterio.open(path) as src:
  rows.append(dict(kind='RGB',mosaic=name,group='',band='RGB',
      path=str(path),width=src.width,height=src.height,
      native_pixel_x_m=abs(src.transform.a),native_pixel_y_m=abs(src.transform.e),
      channels=src.count,dtype=str(src.dtypes[0]),nodata=str(src.nodata),
      crs=str(src.crs),date_tag=str({k:v for k,v in src.tags().items()
                                  if any(w in k.lower() for w in ('date','time','flight'))})[:400]))
for group in GROUP_MOSAIC:
 for band,(path,index) in bands_for(group).items():
  with rasterio.open(path) as src:
   rows.append(dict(kind='MS',mosaic=GROUP_MOSAIC[group],
      group=group,band=band,path=str(path),width=src.width,height=src.height,
      native_pixel_x_m=abs(src.transform.a),native_pixel_y_m=abs(src.transform.e),
      channels=src.count,dtype=str(src.dtypes[index-1]),nodata=str(src.nodata),
      crs=str(src.crs),date_tag=str({k:v for k,v in src.tags().items()
                                  if any(w in k.lower() for w in ('date','time','flight'))})[:400]))
SOURCES=pd.DataFrame(rows);SOURCES.to_csv(OUT/'01_source_raster_properties.csv',index=False)
display(SOURCES[['kind','mosaic','group','band','native_pixel_x_m',
                 'native_pixel_y_m','date_tag']])
acq_cols=['mosaic','group','acquisition_date','flight_speed_m_s','altitude_m',
          'exposure_s','forward_overlap_pct','side_overlap_pct',
          'tiepoint_density','reprojection_RMSE_px','processing_report_path',
          'source_note']
template=SOURCES[['mosaic','group']].drop_duplicates().copy()
for key in acq_cols[2:]:template[key]=pd.NA
metadata_file=OUT/'01_optional_acquisition_metadata_template.csv'
if not metadata_file.is_file():
 template.to_csv(metadata_file,index=False)
print('Optional metadata template:',metadata_file)


## 02 — Cached RGB-only quality proxies and existing joint-sensor coverage

Use one multiband decimated window read per footprint on a persistent mosaic handle, at 0.5 m target spacing. The actual sampling scale is recorded. No MS imagery is read. NB05 joint RGB/DSM/MS valid-pixel counts divided by NB04 valid supervised pixels provide an approximate support fraction, not an MS-only coverage estimate. Cache fingerprints include source metadata, geometry and sampling parameters.


In [ ]:
from rasterio.windows import from_bounds
from rasterio.transform import array_bounds
from scipy.ndimage import binary_erosion
CACHE=OUT/'02_rgb_light_cache_v2';CACHE.mkdir(exist_ok=True)
SUPPORT=pd.read_csv(NB05/'06_all_106_unit_support.csv',dtype={'unit_id':str})
FULL=pd.read_csv(ROOT/'Outputs/04_unified_rgb_heldout_v5/04_heldout_unit_metrics.csv',dtype={'unit_id':str})
assert len(SUPPORT)==len(FULL)==106
assert not SUPPORT.duplicated(['unit_type','unit_id']).any()
assert not FULL.duplicated(['unit_type','unit_id']).any()
assert 'n_pixels' in FULL.columns, FULL.columns.tolist()
SUPPORT=SUPPORT.merge(FULL[['unit_type','unit_id','n_pixels']],on=['unit_type','unit_id'],validate='one_to_one')
SUPPORT['joint_sensor_support_fraction']=SUPPORT.n_common/SUPPORT.n_pixels.replace(0,np.nan)
# Different inference/masking implementations mean small deviations from 1 are possible.
SUPPORT['joint_support_ratio_gt_one']=SUPPORT.joint_sensor_support_fraction.gt(1.001)
def norm_quantile(x,mask):
 v=x[mask&np.isfinite(x)]
 if len(v)<MIN_PIXELS:return np.zeros_like(x,dtype='float32')
 lo,hi=np.percentile(v,[2,98])
 return np.clip((x-lo)/max(hi-lo,1e-6),0,1).astype('float32')
def score_footprint(z,src):
 assert src.crs and src.crs.is_projected and src.crs.linear_units.lower() in ('metre','meter','metres','meters')
 assert abs(src.transform.b)<1e-9 and abs(src.transform.d)<1e-9
 geom=gpd.GeoSeries([z.geometry],crs=UNITS.crs).to_crs(src.crs).iloc[0]
 xmin,ymin,xmax,ymax=geom.bounds
 w=from_bounds(xmin-BUFFER_M,ymin-BUFFER_M,xmax+BUFFER_M,ymax+BUFFER_M,src.transform)
 c0=max(0,int(np.floor(w.col_off)));r0=max(0,int(np.floor(w.row_off)))
 c1=min(src.width,int(np.ceil(w.col_off+w.width)));r1=min(src.height,int(np.ceil(w.row_off+w.height)))
 assert c1>c0 and r1>r0
 win=Window(c0,r0,c1-c0,r1-r0)
 xm=abs(src.transform.a);ym=abs(src.transform.e)
 target=max(SAMPLE_M,win.width*xm/MAX_DIM,win.height*ym/MAX_DIM)
 nx=max(1,int(np.ceil(win.width*xm/target)));ny=max(1,int(np.ceil(win.height*ym/target)))
 tr=src.window_transform(win)*Affine.scale(win.width/nx,win.height/ny)
 inside=rasterize([(geom,1)],out_shape=(ny,nx),transform=tr,fill=0,dtype='uint8').astype(bool)
 bands=[1,2,3]+([4] if src.count>=4 else [])
 # Single multiband read; nearest avoids blending invalid black/white/blue pixels.
 a=src.read(bands,window=win,out_shape=(len(bands),ny,nx),masked=True,resampling=Resampling.nearest)
 rgb=np.asarray(a[:3].filled(0),dtype='float32')
 valid=(~np.ma.getmaskarray(a[:3]).any(axis=0))&np.isfinite(rgb).all(axis=0)
 if len(bands)==4:valid &= (~np.ma.getmaskarray(a[3]))&(a[3].filled(0)>0)
 valid &= np.any(rgb!=0,axis=0)&np.any(rgb!=255,axis=0)
 valid &= ~((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255))
 rv=valid&inside
 gray=rgb[0]*.2126+rgb[1]*.7152+rgb[2]*.0722
 norm=norm_quantile(gray,rv)
 # Exclude invalid-mask boundaries from spatial derivatives.
 interior=binary_erosion(valid,iterations=4)&inside
 smooth=gaussian_filter(norm,1.)
 edges=np.hypot(sobel(smooth,axis=0),sobel(smooth,axis=1))
 lap=np.abs(laplace(gaussian_filter(norm,.65)))
 strong=interior&(edges>=np.percentile(edges[interior],70)) if interior.sum()>=MIN_PIXELS else np.zeros_like(interior)
 v=gray[rv]
 if len(v)>=MIN_PIXELS:
  lo,hi=np.percentile(v,[2,98]);contrast=float(np.std(v[(v>=lo)&(v<=hi)]))
 else:contrast=np.nan
 sat=((rgb<=1).any(axis=0)|(rgb>=254).any(axis=0))&rv
 return dict(site=z.site,unit_type=z.unit_type,unit_id=str(z.unit_id),mosaic=z.mosaic,fold=int(z.fold),
   rgb_native_gsd_m=float(xm),sample_pixel_m=float(abs(tr.a)),sample_pixel_y_m=float(abs(tr.e)),
   reference_pixels=int(inside.sum()),rgb_valid_fraction=float(rv.sum()/max(inside.sum(),1)),
   rgb_gradient_proxy=float(np.median(edges[strong])) if strong.any() else np.nan,
   rgb_laplacian_edge_proxy=float(np.median(lap[strong])) if strong.any() else np.nan,
   rgb_local_contrast=contrast,rgb_saturation_fraction=float(sat.sum()/max(rv.sum(),1)) if rv.any() else np.nan,
   qa_error=None)
rows=[];total=len(UNITS);t0=time.perf_counter()
for mosaic,grp in UNITS.groupby('mosaic',sort=True):
 path=MOSAICS[mosaic];st=path.stat()
 with rasterio.open(path) as src:
  for z in grp.itertuples(index=False):
   key=hashlib.sha256(repr(('rgb_light_v2',str(path),st.st_size,st.st_mtime_ns,z.unit_type,str(z.unit_id),int(z.fold),z.geometry.wkb_hex,str(UNITS.crs),SAMPLE_M,MAX_DIM,BUFFER_M,MIN_PIXELS)).encode()).hexdigest()
   dest=CACHE/(key+'.json');cached=dest.is_file()
   if cached: result=json.loads(dest.read_text())
   else:
    try:
     result=score_footprint(z,src)
     tmp=dest.with_suffix('.tmp');tmp.write_text(json.dumps(result));os.replace(tmp,dest)
    except Exception as exc:
     result=dict(site=z.site,unit_type=z.unit_type,unit_id=str(z.unit_id),mosaic=z.mosaic,fold=int(z.fold),qa_error=str(exc)[:400])
   rows.append(result)
   print(f"[{len(rows)}/{total}] {z.site} {z.unit_type} {z.unit_id} | {'cached' if cached else 'computed'} | elapsed {time.perf_counter()-t0:.1f}s"+(f" | ERROR {result['qa_error']}" if result.get('qa_error') else ''),flush=True)
QUALITY=pd.DataFrame(rows)
for k in ['rgb_native_gsd_m','rgb_gradient_proxy','rgb_laplacian_edge_proxy','rgb_local_contrast','rgb_saturation_fraction','rgb_valid_fraction']:
 if k not in QUALITY:QUALITY[k]=np.nan
QUALITY=QUALITY.merge(SUPPORT[['unit_type','unit_id','joint_sensor_support_fraction','joint_support_ratio_gt_one']],on=['unit_type','unit_id'],validate='one_to_one')
assert len(QUALITY)==len(UNITS)
QUALITY.to_csv(OUT/'02_reference_image_quality_metrics.csv',index=False)
print('Failed RGB reads:',QUALITY.qa_error.notna().sum(),'| support ratios >1.001:',QUALITY.joint_support_ratio_gt_one.sum())
display(QUALITY.groupby('site').agg(n=('unit_id','size'),rgb_GSD_m=('rgb_native_gsd_m','median'),sharpness_proxy=('rgb_laplacian_edge_proxy','median'),joint_support=('joint_sensor_support_fraction','median')).round(3))


## 03 — Join NB05 five-arm held-out errors and create visual diagnostics

**No selection on errors.** This is an explanatory observational analysis, not a new held-out score. The existing 92-unit support is used for paired comparisons. Raw RGB-only 106-unit quality measurements remain separately available.

In [ ]:
PRED_PATH=NB05/'06_all_five_models_matched_unit_metrics.csv'
assert PRED_PATH.is_file(),'Run NB05 Section 06 reporting first'
PRED=pd.read_csv(PRED_PATH)
PRED['unit_id']=PRED.unit_id.astype(str)
REPORT_CONTRACT=json.loads((NB05/'06_reporting_contract.json').read_text())
SUPPORT=pd.read_csv(NB05/'06_all_106_unit_support.csv',dtype={'unit_id':str})
assert len(SUPPORT)==106 and not SUPPORT.duplicated(['unit_type','unit_id']).any()
EXPECTED={'RGB','RGB control','RGB+DSM','RGB+MS','RGB+DSM+MS'}
matched=SUPPORT[SUPPORT.n_common.gt(0)]
assert len(matched)==REPORT_CONTRACT['matched_units']
assert set(PRED.model)==EXPECTED
assert len(PRED)==len(matched)*len(EXPECTED)
assert not PRED.duplicated(['unit_type','unit_id','model']).any()
assert PRED.groupby(['unit_type','unit_id']).model.nunique().eq(len(EXPECTED)).all()
assert set(zip(PRED.unit_type,PRED.unit_id))==set(zip(matched.unit_type,matched.unit_id))
assert PRED.groupby(['unit_type','unit_id']).n_pixels.nunique().eq(1).all()
assert PRED.groupby(['unit_type','unit_id']).observed_cover_pp.nunique().eq(1).all()
if SITE_FILTER: PRED=PRED[PRED.site.eq(SITE_FILTER)].copy()
print('Native V5 predictions:',len(PRED),'rows; models:',sorted(PRED.model.unique()))
JOIN=PRED.merge(QUALITY.drop(columns=['fold','site','mosaic'],errors='ignore'),
            on=['unit_type','unit_id'],validate='many_to_one')
assert len(JOIN)==len(PRED)
JOIN['signed_error_pp']=JOIN.predicted_cover_pp-JOIN.observed_cover_pp
JOIN['absolute_error_pp']=JOIN.signed_error_pp.abs()
JOIN.to_csv(OUT/'03_quality_joined_five_model_errors.csv',index=False)
# Full RGB-only diagnostics remain independent of the joint sensor coverage.
FULL_RGB=pd.read_csv(ROOT/'Outputs/04_unified_rgb_heldout_v5/04_heldout_unit_metrics.csv',dtype={'unit_id':str})
assert len(FULL_RGB)==106 and FULL_RGB.model.eq('unified').all()
if SITE_FILTER: FULL_RGB=FULL_RGB[FULL_RGB.site.eq(SITE_FILTER)].copy()
FULL_JOIN=FULL_RGB.merge(QUALITY.drop(columns=['fold','site','mosaic'],errors='ignore'),on=['unit_type','unit_id'],validate='one_to_one')
assert len(FULL_JOIN)==len(FULL_RGB)
FULL_JOIN['absolute_error_pp']=(FULL_JOIN.predicted_cover_pp-FULL_JOIN.observed_cover_pp).abs()
FULL_JOIN.to_csv(OUT/'03_full_RGB_106_quality_errors.csv',index=False)
KEYS=['rgb_native_gsd_m','rgb_gradient_proxy','rgb_laplacian_edge_proxy','rgb_local_contrast','rgb_saturation_fraction','joint_sensor_support_fraction']
for key in KEYS:
 if key not in JOIN:JOIN[key]=np.nan
p=JOIN[JOIN.model.eq('RGB+MS')].copy()
assert len(p)==len(PRED)//len(EXPECTED)
fig,axs=plt.subplots(2,3,figsize=(14,8),layout='constrained')
for ax,key in zip(axs.flat,[
    'rgb_native_gsd_m','rgb_laplacian_edge_proxy','rgb_local_contrast',
    'joint_sensor_support_fraction','rgb_gradient_proxy','rgb_saturation_fraction']):
 for site,q in p.groupby('site'):
  valid=q[[key,'absolute_error_pp']].replace([np.inf,-np.inf],np.nan).dropna()
  ax.scatter(valid[key],valid.absolute_error_pp,alpha=.7,s=25,label=site)
 ax.set(xlabel=key,ylabel='Absolute RGB+MS cover error (pp)')
 ax.grid(alpha=.15)
axs.flat[0].legend(frameon=False)
fig.suptitle('UAV quality indicators and held-out RGB+MS error (descriptive)')
fig.savefig(FIG/'03_quality_vs_error_scatter.png',dpi=175,bbox_inches='tight')
plt.show()
corr=[]
for site,subset in [('ALL',p),*list(p.groupby('site'))]:
 for key in KEYS:
  q=subset[[key,'absolute_error_pp','observed_cover_pp']].replace(
      [np.inf,-np.inf],np.nan).dropna()
  if len(q)<6 or q[key].nunique()<4:continue
  rho,pval=spearmanr(q[key],q.absolute_error_pp)
  corr.append(dict(site=site,quality_metric=key,n=len(q),
                   rho_spearman=float(rho),p_nominal=float(pval)))
CORR=pd.DataFrame(corr)
CORR.to_csv(OUT/'03_quality_error_associations_exploratory.csv',index=False)
print('Descriptive associations; nominal p values do not correct multiple testing.')
display(CORR.round(3))


## 04 — Control for reference Musanga cover and assess MOS specifically

Small samples and correlated quality metrics limit statistical power. For each metric, report **partial Spearman correlation** between rank-transformed quality and rank-transformed absolute error after residualizing against rank-transformed observed cover. This is a diagnostic, not causal inference. Also show whether model improvements (RGB+MS minus matched fine-tuned RGB control absolute error) covary with quality.

In [ ]:
from scipy.stats import rankdata
from sklearn.linear_model import LinearRegression
def partial_rho(frame,predictor,response='absolute_error_pp',control='observed_cover_pp'):
 q=frame[[predictor,response,control]].replace(
    [np.inf,-np.inf],np.nan).dropna()
 if len(q)<8 or q[predictor].nunique()<4:return np.nan,len(q)
 y=rankdata(q[response]);x=rankdata(q[predictor])
 c=rankdata(q[control]).reshape(-1,1)
 lm=LinearRegression()
 xr=x-lm.fit(c,x).predict(c)
 yr=y-lm.fit(c,y).predict(c)
 if np.std(xr)<1e-8 or np.std(yr)<1e-8:return np.nan,len(q)
 return float(np.corrcoef(xr,yr)[0,1]),len(q)
part=[]
for site,sub in [('ALL',p),*list(p.groupby('site'))]:
 for k in KEYS:
  v,n=partial_rho(sub,k)
  part.append(dict(site=site,quality_metric=k,
                   partial_spearman_given_observed_cover=v,n=n))
PARTIAL=pd.DataFrame(part)
PARTIAL.to_csv(OUT/'04_partial_quality_error_correlations.csv',index=False)
full_part=[]
for site,sub in [('ALL',FULL_JOIN),*list(FULL_JOIN.groupby('site'))]:
 for k in KEYS:
  if k not in sub:continue
  v,n=partial_rho(sub,k)
  full_part.append(dict(site=site,quality_metric=k,partial_spearman_given_observed_cover=v,n=n))
pd.DataFrame(full_part).to_csv(OUT/'04_full_RGB_partial_quality_error_correlations.csv',index=False)
print('MOS cover-adjusted quality-error relationship:')
display(PARTIAL[PARTIAL.site.eq('MOS')].round(3))
pair=JOIN.pivot_table(index=['site','unit_type','unit_id','spatial_group',
   'observed_cover_pp'],columns='model',values='absolute_error_pp').reset_index()
pair['delta_MS_vs_RGB']=pair['RGB+MS']-pair['RGB control']
pair['delta_MS_vs_frozen_RGB']=pair['RGB+MS']-pair['RGB']
qp=pair.merge(QUALITY,on=['unit_type','unit_id'],validate='one_to_one',
               suffixes=('','_qa'))
qp.to_csv(OUT/'04_paired_MS_RGB_quality.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(14,4),layout='constrained')
for ax,key in zip(axes,['rgb_laplacian_edge_proxy','joint_sensor_support_fraction',
                        'rgb_local_contrast']):
 for site,grp in qp.groupby('site'):
  ax.scatter(grp[key],grp.delta_MS_vs_RGB,alpha=.65,s=25,label=site)
 ax.axhline(0,color='0.35',linestyle='--',lw=1)
 ax.set(xlabel=key,ylabel='Δ absolute error: RGB+MS − RGB control (pp)')
 ax.grid(alpha=.12)
axes[0].legend(frameon=False)
fig.suptitle('Does MS help relative to matched RGB control where image quality is poor?')
fig.savefig(FIG/'04_MS_gain_vs_quality.png',dpi=170,bbox_inches='tight')
plt.show()


## 05 — Site summaries, no additional raster reads


In [ ]:
SUMMARY=QUALITY.groupby('site').agg(n=('unit_id','size'),native_gsd_m=('rgb_native_gsd_m','median'),gradient=('rgb_gradient_proxy','median'),contrast=('rgb_local_contrast','median'),joint_support=('joint_sensor_support_fraction','median')).reset_index()
SUMMARY.to_csv(OUT/'05_site_quality_summary.csv',index=False)
display(SUMMARY.round(3))
fig,axes=plt.subplots(1,3,figsize=(12,4),layout='constrained')
for ax,key in zip(axes,['rgb_laplacian_edge_proxy','rgb_local_contrast','joint_sensor_support_fraction']):
 groups=[(site,g[key].dropna().to_numpy()) for site,g in QUALITY.groupby('site')]
 groups=[(site,v) for site,v in groups if len(v)]
 if groups:ax.boxplot([v for _,v in groups],tick_labels=[site for site,_ in groups])
 ax.set_title(key);ax.grid(alpha=.15)
fig.savefig(FIG/'05_site_quality_summary.png',dpi=150,bbox_inches='tight')
plt.show()


## 06 — Interpretation

Texture and contrast are scene-dependent proxies, not measured blur. Joint sensor support is reused from NB05 and is not MS-only coverage. Associations are exploratory and may reflect forest composition, abundance or acquisition differences. No registration-quality or spectral-texture conclusion is supported by this lightweight notebook. No model changes or exclusions.


In [ ]:
manifest=dict(notebook='NB05bis_v5_light_rgb_v2',n_footprints=len(QUALITY),
  n_matched=(len(p) if 'p' in globals() else None),
  indicators=KEYS,site_filter=SITE_FILTER,sample_m=SAMPLE_M,
  source_nb05=str(PRED_PATH),models=sorted(EXPECTED),primary_MS_comparator='RGB control',status='diagnostic_proxies_not_photogrammetry_QA',ms_pixel_reads=False,registration_estimated=False,failed_rgb_reads=int(QUALITY.qa_error.notna().sum()))
(OUT/'05bis_manifest.json').write_text(json.dumps(manifest,indent=2))
print('NB05bis ready. Output:',OUT)
display(pd.DataFrame([dict(file=q.name,size_MB=round(q.stat().st_size/1e6,2))
 for q in sorted(OUT.glob('*.csv'))]))
